In [1]:
import numpy as np
import pandas as pd
import re , string
import matplotlib.pyplot as plt
import seaborn as sns
from preprocessing_text import *
from sklearn.model_selection import train_test_split
from config import *
from dataset_torch import ToxicCommentDataset
from torch.utils.data import  DataLoader
from LSTM_Model import *
from train import *
from evaluation import *
import pickle

c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Softlaptop\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Device: cpu


In [2]:
train_df = pd.read_csv("../Dataset/train.csv")
test_df = pd.read_csv("../Dataset/test.csv")

In [3]:
print("Shape = ",train_df.shape)
train_df.head()

Shape =  (159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [4]:
print("Shape = ",test_df.shape)
test_df.head()

Shape =  (153164, 2)


,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [5]:
LABELS = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

# Dataset info
train_df.info()

# Missing values
print("------------------\n",train_df.isnull().sum())

#duplicated comments
print("------------------\n No of Duplicated Values = ",train_df['comment_text'].duplicated().sum())

print("------------------\n")


<class 'pandas.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   id             159571 non-null  str  
 1   comment_text   159571 non-null  str  
 2   toxic          159571 non-null  int64
 3   severe_toxic   159571 non-null  int64
 4   obscene        159571 non-null  int64
 5   threat         159571 non-null  int64
 6   insult         159571 non-null  int64
 7   identity_hate  159571 non-null  int64
dtypes: int64(6), str(2)
memory usage: 72.2 MB
------------------
 id               0
comment_text     0
toxic            0
severe_toxic     0
obscene          0
threat           0
insult           0
identity_hate    0
dtype: int64
------------------
 No of Duplicated Values =  0
------------------



In [6]:
print("\nClass distribution:")
print(train_df[LABELS].sum())

print("\nClass percentage:")
print((train_df[LABELS].mean() * 100).round(2))


Class distribution:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Class percentage:
toxic            9.58
severe_toxic     1.00
obscene          5.29
threat           0.30
insult           4.94
identity_hate    0.88
dtype: float64


In [7]:
#Preprocessing Text
train_df["clean_text"] = train_df["comment_text"].apply(clean_text)

print(train_df[["comment_text", "clean_text"]].head())

                                        comment_text  \
0  Explanation\nWhy the edits made under my usern...   
1  D'aww! He matches this background colour I'm s...   
2  Hey man, I'm really not trying to edit war. It...   
3  "\nMore\nI can't make any real suggestions on ...   
4  You, sir, are my hero. Any chance you remember...   

                                          clean_text  
0  explanation why the edits made under my userna...  
1  d aww he matches this background colour i am s...  
2  hey man i am really not trying to edit war it ...  
3  more i cannot make any real suggestions on imp...  
4  you sir are my hero any chance you remember wh...  


In [8]:
train_df["tokens"] = train_df["clean_text"].apply(tokenize)

print("\nExample tokens:")
print(train_df["tokens"].iloc[0])


Example tokens:
['explanation', 'why', 'the', 'edits', 'made', 'under', 'my', 'username', 'hardcore', 'metallica', 'fan', 'were', 'reverted', 'they', 'were', 'not', 'vandalisms', 'just', 'closure', 'on', 'some', 'gas', 'after', 'i', 'voted', 'at', 'new', 'york', 'dolls', 'fac', 'and', 'please', 'do', 'not', 'remove', 'the', 'template', 'from', 'the', 'talk', 'page', 'since', 'i', 'am', 'retired', 'now']


In [9]:
# 4. BUILD VOCABULARY
vocab = build_vocabulary(train_df["tokens"])
VOCAB_SIZE = len(vocab)
save_vocab( vocab,VOCAB_PATH)

print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 30000


In [10]:
#SEQUENCE CONVERSION
train_df["sequence"] = train_df["tokens"].apply(lambda x: tokens_to_sequence(x,vocab ))

#PADDING
train_df["padded_sequence"] = train_df["sequence"].apply(lambda x: pad_sequence(x, MAX_LEN))


In [11]:
X = np.array(train_df["padded_sequence"].tolist(),dtype=np.int64)

y = train_df[LABELS].values.astype( np.float32)

print("\nX:", X.shape)
print("Y:", y.shape)


X: (159571, 150)
Y: (159571, 6)


In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y)
model, history = train_model(
    X_train,
    y_train,
    X_val,
    y_val
)


Epoch 1/10 | Train Loss: 0.4576 | Val Loss: 0.3168 | Val F1: 0.4243
Epoch 2/10 | Train Loss: 0.2703 | Val Loss: 0.2585 | Val F1: 0.4263
Epoch 3/10 | Train Loss: 0.2169 | Val Loss: 0.2415 | Val F1: 0.4468


In [ ]:
#LOAD BEST MODEL
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))


<All keys matched successfully>

In [ ]:
# TEST DATASET

test_dataset = ToxicCommentDataset(X_test, y_test)
test_loader = DataLoader(test_dataset,batch_size=BATCH_SIZE, shuffle=False)



In [ ]:
# GET TEST PREDICTIONS

criterion = torch.nn.BCEWithLogitsLoss()

test_loss, y_true, y_probs = validate(model,test_loader,criterion)
best_thresholds, threshold_results = (
    find_optimal_thresholds(
        y_true,
        y_probs,
        LABELS,
        start=0.1,
        end=0.9,
        step=0.05
    )
)

print("\nOptimal thresholds:")
print(threshold_results)


Optimal thresholds:
           Label  Best Threshold   Best F1
0          toxic            0.90  0.797398
1   severe_toxic            0.85  0.495495
2        obscene            0.90  0.809524
3         threat            0.90  0.489796
4         insult            0.90  0.737430
5  identity_hate            0.90  0.421053


In [ ]:
#EVALUATION
results = evaluate_model(y_true,y_probs,threshold=THRESHOLD)



Per-label results:
           Label  Precision    Recall        F1   ROC-AUC    PR-AUC
0          toxic   0.568237  0.920395  0.702662  0.975365  0.882580
1   severe_toxic   0.285132  0.864198  0.428790  0.987679  0.435126
2        obscene   0.581246  0.948598  0.720817  0.988659  0.886034
3         threat   0.236842  0.729730  0.357616  0.977317  0.427419
4         insult   0.479049  0.948020  0.636477  0.983709  0.792501
5  identity_hate   0.214123  0.681159  0.325823  0.975016  0.410993

Classification Report:
               precision    recall  f1-score   support

        toxic       0.57      0.92      0.70      1520
 severe_toxic       0.29      0.86      0.43       162
      obscene       0.58      0.95      0.72       856
       threat       0.24      0.73      0.36        37
       insult       0.48      0.95      0.64       808
identity_hate       0.21      0.68      0.33       138

    micro avg       0.50      0.92      0.65      3521
    macro avg       0.39      0.85    